# GVH–CHH · A1 · Identifiabilité de deux masses
**CHH-COMP-01 v0.1 · Brouillon pré-gel · Contrôle synthétique**  
Référence méthodologique : protocole CHH figé v1.0.

## Niveau 1 — CHH-STRUCTURE et INVERSE-INFERENCE
Deux sous-cubes sont représentés par leurs masses $m_1,m_2>0$ en kg et leurs positions connues $x_1,x_2$ en m sur un axe. Les inconnues sont les masses. Le test porte sur la composition statique et l'inférence inverse. Le jacobien utilisé est une sensibilité aux paramètres de mesure. Aucune équation de mouvement n'intervient.

Les jeux synthétiques d'évaluation sont générés avec une graine fixée. Les estimateurs sont des formules analytiques définies avant génération et ne reçoivent jamais les masses de référence.


## Niveau 2 — KNOWN-PHYSICS
La masse totale et le barycentre vérifient
$$M=m_1+m_2,\qquad x_c=\frac{m_1x_1+m_2x_2}{M}.$$
La masse seule donne un jacobien $[1,1]$ de rang 1. Pour $M>0$, le passage de $(M,x_c)$ à $(M,Mx_c)$ est inversible. Ce dernier couple est linéaire en $(m_1,m_2)$, de matrice
$$A=\begin{pmatrix}1&1\\x_1&x_2\end{pmatrix},\qquad \det A=x_2-x_1.$$
Le rang vaut 2 si les positions diffèrent et 1 si elles coïncident. Pour $d=x_2-x_1\ne0$,
$$m_1=M\frac{x_2-x_c}{d},\qquad m_2=M\frac{x_c-x_1}{d}.$$
Avec $M$ exact et une erreur $\epsilon$ additive sur $x_c$, les erreurs sont exactement $-M\epsilon/d$ et $+M\epsilon/d$. Si $M$ est aussi bruitée, l'estimateur applique la même inversion au couple mesuré. Une masse reconstruite négative signale une incompatibilité avec le domaine physique. Le test conserve ces estimations dans les métriques.


## Niveau 3 — NUMERICAL-TEST
Contrôles fixés dans cette version du notebook :

1. Deux couples distincts produisent la même masse totale.
2. Avec positions distinctes et mesures exactes, retrouver les masses sur 2 000 systèmes à une tolérance de $10^{-10}$ kg.
3. Aux positions coïncidentes, constater la dégénérescence et retourner l'absence de solution unique.
4. Sur les mêmes systèmes et les mêmes tirages de bruit, vérifier que la RMSE du cas $M$ exact croît comme $1/|d|$. Avec les deux mesures bruitées, comparer la RMSE entre la séparation la plus grande et la plus petite.

Paramètres simulés : masses uniformes dans [2, 8] kg, bruits gaussiens indépendants de $\sigma_M=0{,}05$ kg et $\sigma_{x_c}=0{,}01$ m. Les positions sont exactes. Le comparateur masse seule répartit la masse également. Ce choix de référence représente une hypothèse supplémentaire de l'estimateur.


In [1]:
from math import isclose, sqrt
from random import Random
from statistics import fmean

def barycentre(m1, m2, x1, x2):
    return (m1*x1 + m2*x2) / (m1+m2)

def reconstruire(masse, centre, x1, x2):
    separation = x2 - x1
    if separation == 0:
        return None
    m2 = masse * (centre - x1) / separation
    return masse - m2, m2

def rmse(predictions, references):
    return sqrt(fmean(
        (estimation - vraie)**2
        for prediction, reference in zip(predictions, references, strict=True)
        for estimation, vraie in zip(prediction, reference, strict=True)
    ))

rng = Random(20261120)
masses = [(rng.uniform(2, 8), rng.uniform(2, 8)) for _ in range(2000)]
bruits = [(rng.gauss(0, 0.05), rng.gauss(0, 0.01)) for _ in masses]

assert sum((2, 8)) == sum((4, 6))
assert barycentre(2, 8, 3, 3) == barycentre(4, 6, 3, 3)
assert reconstruire(10, 3, 3, 3) is None
erreur_max = max(
    abs(estimateur - vraie)
    for m1, m2 in masses
    for estimateur, vraie in zip(
        reconstruire(m1+m2, barycentre(m1, m2, 0, 1), 0, 1), (m1, m2)
    )
)
assert erreur_max < 1e-10
print(f"Erreur maximale sans bruit : {erreur_max:.3e} kg")
print("Rangs analytiques : 1 (M seul), 2 (M et barycentre), 1 (coïncidence).")


Erreur maximale sans bruit : 1.776e-15 kg
Rangs analytiques : 1 (M seul), 2 (M et barycentre), 1 (coïncidence).


In [2]:
reference = [((m1+m2)/2, (m1+m2)/2) for m1, m2 in masses]
print(f"RMSE du comparateur masse seule à parts égales : {rmse(reference, masses):.4f} kg")
print(f"{'d (m)':>7} {'RMSE M exact (kg)':>19} {'RMSE deux bruits (kg)':>24} {'m < 0 (%)':>12}")
erreurs = []
for separation in (2.0, 0.5, 0.1, 0.02):
    masses_exactes, deux_bruits = [], []
    for (m1, m2), (delta_m, delta_x) in zip(masses, bruits, strict=True):
        centre_bruite = barycentre(m1, m2, 0, separation) + delta_x
        masses_exactes.append(reconstruire(m1+m2, centre_bruite, 0, separation))
        deux_bruits.append(reconstruire(m1+m2+delta_m, centre_bruite, 0, separation))
    rmse_exacte = rmse(masses_exactes, masses)
    rmse_bruitee = rmse(deux_bruits, masses)
    rmse_theorique = sqrt(fmean(
        ((m1+m2)*delta_x/separation)**2
        for (m1, m2), (_, delta_x) in zip(masses, bruits, strict=True)
    ))
    assert isclose(rmse_exacte, rmse_theorique, rel_tol=1e-12)
    erreurs.append((rmse_exacte, rmse_bruitee))
    taux_negatif = 100 * fmean(min(p) < 0 for p in deux_bruits)
    print(f"{separation:7.2f} {rmse_exacte:19.4f} {rmse_bruitee:24.4f} {taux_negatif:12.2f}")
assert isclose(erreurs[-1][0]/erreurs[0][0], 100, rel_tol=1e-12)
assert erreurs[-1][1] > erreurs[0][1]
print("Verdict numérique : PASS_SYNTHETIC_TWO_MASS_IDENTIFIABILITY")


RMSE du comparateur masse seule à parts égales : 1.2381 kg
  d (m)   RMSE M exact (kg)    RMSE deux bruits (kg)    m < 0 (%)
   2.00              0.0510                   0.0572         0.00
   0.50              0.2040                   0.2056         0.00
   0.10              1.0201                   1.0204         0.10
   0.02              5.1005                   5.1008        33.20
Verdict numérique : PASS_SYNTHETIC_TWO_MASS_IDENTIFIABILITY


## Niveau 4 — Verdict et limites
La somme détermine uniquement la masse totale. Le barycentre fournit une information indépendante si les positions sont distinctes. Le problème reste identifiable en théorie lorsque la séparation est faible, mais la reconstruction se dégrade avec le bruit. À séparation nulle, l'information manque même en l'absence de bruit.

Portée du PASS : cas synthétique à deux masses, positions exactes et modèle additif de référence. La validation empirique reste ouverte. Pour passer aux données réelles, il faudra documenter comment le barycentre est obtenu, sa covariance avec la masse totale et la précision des positions. Les variantes non additives et les cas à plus de deux sous-cubes restent à étudier.
